# ROGII EDA

This notebook explores the ROGII Wellbore Geology Prediction data before modeling. The focus is the file-per-well structure, hidden `TVT_input` intervals, train/test schema differences, and the evidence for trajectory-style modeling.

Workflow:

1. Resolve the Kaggle data path and inventory files.
2. Parse `sample_submission.csv` and validate row-index targets.
3. Summarize horizontal wells and paired typewells.
4. Compare train/test schemas and missingness.
5. Inspect representative well logs and typewell coverage.
6. Summarize modeling implications for the baseline notebook.


## 1. Setup

The next code cell defines display options, Viridis plot defaults, and Kaggle data-root discovery for the exploratory run.


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Callable, Optional, Sequence
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, Markdown, display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
sns.set_theme(style="whitegrid", context="notebook", palette="viridis")
VIRIDIS = sns.color_palette("viridis", as_cmap=True)
VIRIDIS_COLORS = sns.color_palette("viridis", 8)


class CFG:
    """Notebook runtime configuration."""

    MODE = "eda"


RUN_MODE = CFG.MODE

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
COMPETITION_SLUG = "rogii-wellbore-geology-prediction"
DATA_ROOT_CANDIDATES = [
    KAGGLE_INPUT_ROOT / "competitions" / COMPETITION_SLUG,
    KAGGLE_INPUT_ROOT / COMPETITION_SLUG,
]


def resolve_data_root(candidates: Sequence[Path]) -> Path:
    """Resolve the Kaggle competition data directory.

    Args:
        candidates (Sequence[Path]): Candidate data-root paths.

    Returns:
        Path: Resolved path.
    """
    for candidate in candidates:
        if (candidate / "sample_submission.csv").exists():
            return candidate
    for sample_file in (
        KAGGLE_INPUT_ROOT.rglob("sample_submission.csv")
        if KAGGLE_INPUT_ROOT.exists()
        else []
    ):
        if COMPETITION_SLUG in sample_file.as_posix():
            return sample_file.parent
    return candidates[0]


DATA_ROOT = resolve_data_root(DATA_ROOT_CANDIDATES)
print("Kaggle input root:", KAGGLE_INPUT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("Exists:", DATA_ROOT.exists())
print(
    "sample_submission exists:", (DATA_ROOT / "sample_submission.csv").exists()
)


def fmt_int(value: object) -> str:
    """Format an integer-like value for markdown output.

    Args:
        value (object): Value to parse or format.

    Returns:
        str: Formatted string.
    """
    if pd.isna(value):
        return "n/a"
    return f"{int(value):,}"


def fmt_float(value: object, digits: int = 2) -> str:
    """Format a float-like value for markdown output.

    Args:
        value (object): Value to parse or format.
        digits (int): Number of decimal places.

    Returns:
        str: Formatted string.
    """
    if pd.isna(value):
        return "n/a"
    return f"{float(value):,.{digits}f}"


def fmt_pct(value: object, digits: int = 1) -> str:
    """Format a proportion as a percentage string.

    Args:
        value (object): Value to parse or format.
        digits (int): Number of decimal places.

    Returns:
        str: Formatted string.
    """
    if pd.isna(value):
        return "n/a"
    return f"{100 * float(value):,.{digits}f}%"


def show_insights(title: str, bullets: Sequence[object]) -> None:
    """Display a compact markdown insight block.

    Args:
        title (str): Insight block title.
        bullets (Sequence[object]): Insight bullet strings.

    Returns:
        None: This function updates state or displays output.
    """
    clean_bullets = [str(bullet) for bullet in bullets if bullet]
    body = "\n".join(f"- {bullet}" for bullet in clean_bullets)
    display(Markdown(f"### {title}\n{body}"))


## 2. Data Discovery

This section confirms that Kaggle mounted the competition data correctly, inventories the available files, and parses the submission key. The competition uses a nested file layout:

- horizontal wells contain measured depth (`MD`), coordinates, gamma ray (`GR`), and `TVT_input`;
- typewells provide the vertical `GR` signature indexed by `TVT`, plus geology labels;
- `sample_submission.csv` identifies the exact horizontal-well rows that require prediction.

The submission key has the form `{WELLNAME}_{row_index}`. Splitting it into `well` and `row_idx` gives an early check that requested predictions align with the hidden suffix of each horizontal well.

In the saved public-sample run, the dataset contains 773 training horizontal wells, 773 training typewells, 3 public test horizontal wells, 3 public test typewells, and 773 PNG reference images. The hidden test rerun can expose more wells, so the notebook avoids hardcoded well IDs.

In [ ]:
def find_files(root: Path, pattern: str) -> list[Path]:
    """Return sorted files matching a recursive pattern.

    Args:
        root (Path): Directory to search.
        pattern (str): Recursive glob pattern.

    Returns:
        list[Path]: Matching paths.
    """
    return sorted(root.rglob(pattern)) if root.exists() else []


def well_name_from_horizontal_path(path: Path) -> str:
    """Extract a well name from a horizontal-well path.

    Args:
        path (Path): Input file path.

    Returns:
        str: Formatted string.
    """
    return path.name.split("__horizontal_well.csv")[0]


def well_name_from_typewell_path(path: Path) -> str:
    """Extract a well name from a typewell path.

    Args:
        path (Path): Input file path.

    Returns:
        str: Formatted string.
    """
    return path.name.split("__typewell.csv")[0]


def parse_submission_id(value: object) -> tuple[str, int]:
    """Split a submission id into well name and row index.

    Args:
        value (object): Value to parse or format.

    Returns:
        tuple[str, int]: Well name and row index.
    """
    well, row = str(value).rsplit("_", 1)
    return well, int(row)


def get_column(df: pd.DataFrame, name: str) -> Optional[str]:
    """Return the matching DataFrame column name, ignoring case.

    Args:
        df (pd.DataFrame): Input DataFrame.
        name (str): Column or model name.

    Returns:
        Optional[str]: Computed result.
    """
    lookup = {col.lower(): col for col in df.columns}
    return lookup.get(name.lower())


train_dir = DATA_ROOT / "train"
test_dir = DATA_ROOT / "test"
sample_path = DATA_ROOT / "sample_submission.csv"

train_horizontal_files = find_files(train_dir, "*__horizontal_well.csv")
train_typewell_files = find_files(train_dir, "*__typewell.csv")
test_horizontal_files = find_files(test_dir, "*__horizontal_well.csv")
test_typewell_files = find_files(test_dir, "*__typewell.csv")
png_files = find_files(DATA_ROOT, "*.png")

inventory = pd.DataFrame(
    [
        {
            "split": "train",
            "file_type": "horizontal_well",
            "count": len(train_horizontal_files),
        },
        {
            "split": "train",
            "file_type": "typewell",
            "count": len(train_typewell_files),
        },
        {
            "split": "test",
            "file_type": "horizontal_well",
            "count": len(test_horizontal_files),
        },
        {
            "split": "test",
            "file_type": "typewell",
            "count": len(test_typewell_files),
        },
        {"split": "all", "file_type": "png", "count": len(png_files)},
        {
            "split": "all",
            "file_type": "sample_submission",
            "count": int(sample_path.exists()),
        },
    ]
)
display(inventory)
print(
    "First train wells:",
    [well_name_from_horizontal_path(p) for p in train_horizontal_files[:5]],
)
print(
    "First test wells:",
    [well_name_from_horizontal_path(p) for p in test_horizontal_files[:5]],
)


In [ ]:
if "inventory" in globals() and not inventory.empty:
    counts = inventory.set_index(["split", "file_type"])["count"]
    train_wells = counts.get(("train", "horizontal_well"), 0)
    test_wells = counts.get(("test", "horizontal_well"), 0)
    png_count = counts.get(("all", "png"), 0)
    show_insights(
        "Discovery Insights",
        [
            (
                f"Train inventory contains {fmt_int(train_wells)} horizontal "
                "wells and matching typewell files, giving a broad set "
                "of well-level examples for validation."
            ),
            (
                f"Public test inventory contains {fmt_int(test_wells)} "
                "horizontal wells. Kaggle code competitions can rerun "
                "on a larger hidden test set, so all downstream logic "
                "is file-discovery based."
            ),
            (
                f"The {fmt_int(png_count)} PNG reference images are useful "
                "for qualitative QA, but the notebook keeps modeling "
                "inputs to CSV features that are available at inference time."
            ),
            (
                f"Data root resolved to `{DATA_ROOT}`; "
                "`sample_submission.csv` exists: "
                f"{(DATA_ROOT / 'sample_submission.csv').exists()}."
            ),
        ],
    )


In [ ]:
sample_submission = pd.read_csv(sample_path)
id_col = sample_submission.columns[0]
target_col = (
    "tvt"
    if "tvt" in sample_submission.columns
    else sample_submission.columns[-1]
)

submission_index = sample_submission[id_col].map(parse_submission_id)
sample_submission["well"] = [x[0] for x in submission_index]
sample_submission["row_idx"] = [x[1] for x in submission_index]

display(sample_submission.head())
print("sample_submission shape:", sample_submission.shape)
display(
    sample_submission.groupby("well")["row_idx"]
    .agg(["min", "max", "count"])
    .head(10)
)


In [ ]:
if "sample_submission" in globals() and not sample_submission.empty:
    sub_summary = sample_submission.groupby("well")["row_idx"].agg(
        ["min", "max", "count"]
    )
    show_insights(
        "Submission Index Insights",
        [
            (
                f"The public sample requests {fmt_int(len(sample_submission))} "
                "predictions across "
                f"{fmt_int(sample_submission['well'].nunique())} wells."
            ),
            (
                "Per-well requested rows range from "
                f"{fmt_int(sub_summary['count'].min())} to "
                f"{fmt_int(sub_summary['count'].max())}, so the hidden "
                "interval length varies by well."
            ),
            (
                "The first requested row is usually the first hidden "
                "`TVT_input` row; mismatches here would signal an "
                "indexing or path issue before submission generation."
            ),
        ],
    )


## 3. Well Metadata

This section reads each horizontal well and typewell once to build compact, well-level summaries. The goal is not to model yet; it is to understand the shape of the problem with enough detail to avoid mistakes later.

Key fields captured here:

- row counts and measured-depth ranges, which describe lateral length;
- `GR` mean and standard deviation, which show how log character varies across wells;
- `TVT_input` known and missing counts, which reveal the size of the hidden interval;
- train-only `TVT` range, which gives a rough sense of target movement by well;
- typewell `TVT` range and geology label counts, which describe the reference logs used for correlation.

In the saved public-sample run, the public test wells are actually drawn from train-like examples, making them useful for notebook validation but not necessarily representative of the hidden leaderboard set.

In [ ]:
def summarize_horizontal_file(path: Path, split: str) -> dict[str, object]:
    """Build one metadata row for a horizontal-well file.

    Args:
        path (Path): Input file path.
        split (str): Dataset split label.

    Returns:
        dict[str, object]: Computed result.
    """
    well = well_name_from_horizontal_path(path)
    df = pd.read_csv(path)
    tvt_input_col = get_column(df, "TVT_input")
    tvt_col = get_column(df, "TVT")
    md_col = get_column(df, "MD")
    gr_col = get_column(df, "GR")

    row = {
        "split": split,
        "well": well,
        "rows": len(df),
        "n_columns": df.shape[1],
        "columns": tuple(df.columns),
        "has_tvt": tvt_col is not None,
        "has_tvt_input": tvt_input_col is not None,
        "md_min": (
            pd.to_numeric(df[md_col], errors="coerce").min()
            if md_col
            else np.nan
        ),
        "md_max": (
            pd.to_numeric(df[md_col], errors="coerce").max()
            if md_col
            else np.nan
        ),
        "gr_mean": (
            pd.to_numeric(df[gr_col], errors="coerce").mean()
            if gr_col
            else np.nan
        ),
        "gr_std": (
            pd.to_numeric(df[gr_col], errors="coerce").std()
            if gr_col
            else np.nan
        ),
    }

    if tvt_input_col:
        tvt_input = pd.to_numeric(df[tvt_input_col], errors="coerce")
        hidden_mask = tvt_input.isna()
        row["tvt_input_known"] = int(tvt_input.notna().sum())
        row["tvt_input_missing"] = int(hidden_mask.sum())
        row["tvt_input_missing_frac"] = float(hidden_mask.mean())
        row["first_missing_row"] = (
            int(np.argmax(hidden_mask.to_numpy()))
            if hidden_mask.any()
            else np.nan
        )
        row["last_known_tvt_input"] = (
            float(tvt_input.ffill().iloc[-1])
            if tvt_input.notna().any()
            else np.nan
        )
    else:
        row["tvt_input_known"] = 0
        row["tvt_input_missing"] = np.nan
        row["tvt_input_missing_frac"] = np.nan
        row["first_missing_row"] = np.nan
        row["last_known_tvt_input"] = np.nan

    if tvt_col:
        tvt = pd.to_numeric(df[tvt_col], errors="coerce")
        row["tvt_min"] = tvt.min()
        row["tvt_max"] = tvt.max()
        row["tvt_range"] = tvt.max() - tvt.min()
    else:
        row["tvt_min"] = np.nan
        row["tvt_max"] = np.nan
        row["tvt_range"] = np.nan

    return row


def summarize_typewell_file(path: Path, split: str) -> dict[str, object]:
    """Build one metadata row for a typewell file.

    Args:
        path (Path): Input file path.
        split (str): Dataset split label.

    Returns:
        dict[str, object]: Computed result.
    """
    well = well_name_from_typewell_path(path)
    df = pd.read_csv(path)
    tvt_col = get_column(df, "TVT")
    gr_col = get_column(df, "GR")
    geology_col = get_column(df, "Geology")
    return {
        "split": split,
        "well": well,
        "rows": len(df),
        "n_columns": df.shape[1],
        "columns": tuple(df.columns),
        "tvt_min": (
            pd.to_numeric(df[tvt_col], errors="coerce").min()
            if tvt_col
            else np.nan
        ),
        "tvt_max": (
            pd.to_numeric(df[tvt_col], errors="coerce").max()
            if tvt_col
            else np.nan
        ),
        "gr_mean": (
            pd.to_numeric(df[gr_col], errors="coerce").mean()
            if gr_col
            else np.nan
        ),
        "gr_std": (
            pd.to_numeric(df[gr_col], errors="coerce").std()
            if gr_col
            else np.nan
        ),
        "n_geology_labels": (
            df[geology_col].nunique() if geology_col else np.nan
        ),
    }


horizontal_meta = pd.DataFrame(
    [summarize_horizontal_file(p, "train") for p in train_horizontal_files]
    + [summarize_horizontal_file(p, "test") for p in test_horizontal_files]
)
typewell_meta = pd.DataFrame(
    [summarize_typewell_file(p, "train") for p in train_typewell_files]
    + [summarize_typewell_file(p, "test") for p in test_typewell_files]
)

display(horizontal_meta.head())
display(typewell_meta.head())


In [ ]:
if "horizontal_meta" in globals() and not horizontal_meta.empty:
    train_meta = horizontal_meta.query("split == 'train'")
    test_meta = horizontal_meta.query("split == 'test'")
    bullets = []
    if not train_meta.empty:
        bullets.append(
            (
                f"Train wells have a median of "
                f"{fmt_int(train_meta['rows'].median())} horizontal rows "
                "and a median hidden `TVT_input` fraction of "
                f"{fmt_pct(train_meta['tvt_input_missing_frac'].median())}."
            )
        )
        bullets.append(
            (
                "Train `TVT` range varies substantially by well: "
                f"median range {fmt_float(train_meta['tvt_range'].median())} "
                f"ft, max range {fmt_float(train_meta['tvt_range'].max())} ft."
            )
        )
    if not test_meta.empty:
        bullets.append(
            (
                f"Public test wells have a median of "
                f"{fmt_int(test_meta['rows'].median())} rows and "
                "median hidden fraction of "
                f"{fmt_pct(test_meta['tvt_input_missing_frac'].median())}."
            )
        )
    if "typewell_meta" in globals() and not typewell_meta.empty:
        bullets.append(
            (
                "Typewells provide compact vertical references with median "
                f"{fmt_int(typewell_meta['rows'].median())} rows and "
                f"median {fmt_int(typewell_meta['n_geology_labels'].median())} "
                "geology labels."
            )
        )
    show_insights("Well Metadata Insights", bullets)


## 4. Schema And Missingness

This section checks which columns are available in train versus test, summarizes missingness, and compares well-level distributions. This is the most important leakage-control step in the notebook.

Observed from the saved public-sample run:

- train horizontal wells include `TVT` plus geology-top columns such as `ANCC`, `ASTNU`, `ASTNL`, `EGFDU`, `EGFDL`, and `BUDA`;
- test horizontal wells expose only inference-time columns such as `MD`, `X`, `Y`, `Z`, `GR`, and `TVT_input`;
- approximately 73% of each horizontal well is hidden in `TVT_input`, so the task is closer to sequence continuation / log correlation than ordinary row-wise regression.

The distribution plots compare well length, hidden-window size, and typewell coverage across splits. They are quick checks for public-test representativeness and for validation design: if test wells sit in the same range as train wells, masked-tail validation is more trustworthy.

Modeling implication: train-only geology-top columns should not be used directly as features for a final test-time model unless they are predicted by a separate model that can also run on test data.

In [ ]:
def column_presence(
    files: Sequence[Path], split: str, well_parser: Callable[[Path], str]
) -> pd.DataFrame:
    """Summarize column availability across well files.

    Args:
        files (Sequence[Path]): Input well files.
        split (str): Dataset split label.
        well_parser (Callable[[Path], str]): Function that extracts a well name from a path.

    Returns:
        pd.DataFrame: Computed DataFrame.
    """
    rows = []
    for path in files:
        well = well_parser(path)
        df = pd.read_csv(path, nrows=5)
        for col in df.columns:
            rows.append({"split": split, "well": well, "column": col})
    return pd.DataFrame(rows)


horizontal_columns = pd.concat(
    [
        column_presence(
            train_horizontal_files, "train", well_name_from_horizontal_path
        ),
        column_presence(
            test_horizontal_files, "test", well_name_from_horizontal_path
        ),
    ],
    ignore_index=True,
)

column_summary = (
    horizontal_columns.groupby(["split", "column"])["well"]
    .nunique()
    .reset_index(name="well_count")
    .sort_values(
        ["split", "well_count", "column"], ascending=[True, False, True]
    )
)
display(column_summary)

if not horizontal_meta.empty:
    display(
        horizontal_meta.groupby("split")[
            [
                "rows",
                "tvt_input_missing_frac",
                "tvt_range",
                "gr_mean",
                "gr_std",
            ]
        ]
        .describe()
        .T
    )


In [ ]:
if not horizontal_meta.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    sns.histplot(
        data=horizontal_meta,
        x="rows",
        hue="split",
        bins=30,
        ax=axes[0],
        element="step",
        palette="viridis",
    )
    axes[0].set_title("Rows per horizontal well")
    sns.histplot(
        data=horizontal_meta,
        x="tvt_input_missing_frac",
        hue="split",
        bins=30,
        ax=axes[1],
        element="step",
        palette="viridis",
    )
    axes[1].set_title("Hidden TVT_input fraction")
    sns.scatterplot(
        data=horizontal_meta,
        x="rows",
        y="tvt_input_missing",
        hue="split",
        ax=axes[2],
        palette="viridis",
    )
    axes[2].set_title("Hidden rows by well length")
    plt.tight_layout()
    plt.show()

if not typewell_meta.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.histplot(
        data=typewell_meta,
        x="rows",
        hue="split",
        bins=30,
        ax=axes[0],
        element="step",
        palette="viridis",
    )
    axes[0].set_title("Rows per typewell")
    sns.histplot(
        data=typewell_meta,
        x="n_geology_labels",
        hue="split",
        bins=20,
        ax=axes[1],
        element="step",
        palette="viridis",
    )
    axes[1].set_title("Geology labels per typewell")
    plt.tight_layout()
    plt.show()


In [ ]:
if "horizontal_meta" in globals() and not horizontal_meta.empty:
    by_split = horizontal_meta.groupby("split")[
        "tvt_input_missing_frac"
    ].median()
    show_insights(
        "Missingness And Window Insights",
        [
            f"Median hidden `TVT_input` fraction by split: "
            + ", ".join(
                [f"{idx}: {fmt_pct(val)}" for idx, val in by_split.items()]
            )
            + ".",
            (
                "The hidden interval is long enough that pure carry-forward "
                "is a weak geological model; it is mainly a submission "
                "smoke test."
            ),
            (
                "Because test lacks `TVT` and formation-top columns, final "
                "models should use inference-available features or auxiliary "
                "predictions that are also generated for test."
            ),
        ],
    )


## 5. Example Well

The competition provides PNG reference images for many wells. These are useful for human QA: they show the horizontal path and geological context, while the extracted metadata shows how the same well appears in the CSV files.

This section displays one representative public-sample well image next to metadata extracted from the horizontal well, typewell, and submission index.

In [ ]:
png_lookup = {path.stem: path for path in png_files}

if "sample_submission" in globals() and not sample_submission.empty:
    example_well = sample_submission["well"].iloc[0]
elif not horizontal_meta.empty:
    example_well = horizontal_meta["well"].iloc[0]
else:
    example_well = None

if example_well is not None:
    image_path = png_lookup.get(example_well)
    horizontal_row = horizontal_meta.query("well == @example_well").drop(
        columns=["columns"], errors="ignore"
    )
    typewell_row = typewell_meta.query("well == @example_well").drop(
        columns=["columns"], errors="ignore"
    )
    submission_row = (
        sample_submission.query("well == @example_well")["row_idx"]
        .agg(["min", "max", "count"])
        .to_frame(name="submission_rows")
        .T
        if "sample_submission" in globals()
        and example_well in set(sample_submission["well"])
        else pd.DataFrame()
    )

    show_insights(
        "Example Well Image Readout",
        [
            f"Example well: `{example_well}`.",
            f"PNG image found: {image_path is not None}.",
            (
                "Use this visual as qualitative context only; current "
                "models remain CSV-feature based."
            ),
        ],
    )
    display(horizontal_row)
    display(typewell_row)
    display(submission_row)
    if image_path is not None:
        display(Image(filename=str(image_path), width=1000))
else:
    print("No example well available.")


## 6. Log Inspection

These plots inspect a few evaluation wells at the well-log level. Each row of plots is designed to answer a modeling question:

- Does the horizontal `GR` curve contain recognizable peaks and troughs that can be aligned to the typewell?
- Where does `TVT_input` stop, and how long is the hidden interval?
- Does the typewell cover the same `TVT` depth range needed by the horizontal well?

The visual pattern to look for is local similarity between the horizontal `GR` trace and the typewell `GR` trace after mapping the horizontal row/MD position to TVT. This is the domain reason that alignment-style models can be more promising than plain tabular models.

In [ ]:
horizontal_lookup = {
    **{well_name_from_horizontal_path(p): p for p in train_horizontal_files},
    **{well_name_from_horizontal_path(p): p for p in test_horizontal_files},
}
typewell_lookup = {
    **{well_name_from_typewell_path(p): p for p in train_typewell_files},
    **{well_name_from_typewell_path(p): p for p in test_typewell_files},
}


def load_well(well: str) -> tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    """Load the horizontal well and paired typewell for one well.

    Args:
        well (str): Well identifier.

    Returns:
        tuple[pd.DataFrame, Optional[pd.DataFrame]]: Computed result.
    """
    horizontal = pd.read_csv(horizontal_lookup[well])
    typewell = (
        pd.read_csv(typewell_lookup[well]) if well in typewell_lookup else None
    )
    return horizontal, typewell


def plot_well(well: str) -> None:
    """Plot representative horizontal and typewell logs.

    Args:
        well (str): Well identifier.

    Returns:
        None: This function updates state or displays output.
    """
    horizontal, typewell = load_well(well)
    md_col = get_column(horizontal, "MD")
    gr_col = get_column(horizontal, "GR")
    tvt_input_col = get_column(horizontal, "TVT_input")
    tvt_col = get_column(horizontal, "TVT")

    x = (
        pd.to_numeric(horizontal[md_col], errors="coerce")
        if md_col
        else horizontal.index
    )
    fig, axes = plt.subplots(1, 3, figsize=(19, 4))

    if gr_col:
        axes[0].plot(
            x,
            pd.to_numeric(horizontal[gr_col], errors="coerce"),
            lw=1,
            color=VIRIDIS_COLORS[2],
        )
    axes[0].set_title(f"{well}: horizontal GR")
    axes[0].set_xlabel("MD" if md_col else "row")
    axes[0].set_ylabel("GR")

    if tvt_input_col:
        tvt_input = pd.to_numeric(horizontal[tvt_input_col], errors="coerce")
        axes[1].plot(
            x, tvt_input, lw=1, label="TVT_input", color=VIRIDIS_COLORS[4]
        )
        if tvt_input.isna().any():
            hidden_start = int(np.argmax(tvt_input.isna().to_numpy()))
            axes[1].axvline(
                x.iloc[hidden_start] if hasattr(x, "iloc") else hidden_start,
                color=VIRIDIS_COLORS[7],
                ls="--",
                lw=1,
                label="first hidden row",
            )
    if tvt_col:
        axes[1].plot(
            x,
            pd.to_numeric(horizontal[tvt_col], errors="coerce"),
            lw=1,
            alpha=0.65,
            label="TVT",
            color=VIRIDIS_COLORS[1],
        )
    axes[1].invert_yaxis()
    axes[1].set_title("Horizontal TVT track")
    axes[1].set_xlabel("MD" if md_col else "row")
    axes[1].legend(loc="best")

    if typewell is not None:
        tw_tvt_col = get_column(typewell, "TVT")
        tw_gr_col = get_column(typewell, "GR")
        geology_col = get_column(typewell, "Geology")
        if tw_tvt_col and tw_gr_col:
            axes[2].plot(
                pd.to_numeric(typewell[tw_gr_col], errors="coerce"),
                pd.to_numeric(typewell[tw_tvt_col], errors="coerce"),
                lw=1,
                color=VIRIDIS_COLORS[3],
            )
            axes[2].invert_yaxis()
        if geology_col:
            label_counts = typewell[geology_col].value_counts().head(8)
            axes[2].text(
                0.98,
                0.02,
                "\n".join([f"{k}: {v}" for k, v in label_counts.items()]),
                transform=axes[2].transAxes,
                ha="right",
                va="bottom",
                fontsize=8,
            )
    axes[2].set_title("Typewell GR vs TVT")
    axes[2].set_xlabel("GR")
    axes[2].set_ylabel("TVT")

    plt.tight_layout()
    plt.show()


sample_wells = list(sample_submission["well"].drop_duplicates().head(3))
if len(sample_wells) < 3:
    sample_wells += [w for w in horizontal_lookup if w not in sample_wells][
        : 3 - len(sample_wells)
    ]

for well in sample_wells:
    if well in horizontal_lookup:
        plot_well(well)


## 7. Target Relationships

This section samples training rows to study numeric relationships without forcing the whole dataset into memory at once. The focus is on `TVT`, `TVT_input`, `MD`, `Z`, `GR`, and the train-only formation-top columns.

The target distribution and scatter checks show coarse relationships between `TVT`, `GR`, and `MD`. The goal is not to prove a linear model; it is to see whether simple global relationships exist or whether the problem needs per-well, sequence-aware features.

Saved-run observations:

- the training sample contains 90,000 rows from 60 wells;
- `TVT_input` is populated only in the observed part of each well, so it has far fewer non-null sampled rows than `TVT`;
- `GR` has substantial missingness in the sampled rows, so models need an explicit missing-data strategy;
- formation-top columns are highly informative in train but absent from test, making them a leakage risk if used naively.

Useful next modeling directions include lag/rolling features over `GR`, slope features over `TVT_input`, typewell-correlation features, and per-well normalization of log values.

In [ ]:
def load_horizontal_sample(
    files: Sequence[Path],
    max_wells: int = 60,
    rows_per_well: int = 1500,
    split: str = "train",
) -> pd.DataFrame:
    """Load a bounded sample of horizontal-well rows.

    Args:
        files (Sequence[Path]): Input well files.
        max_wells (int): Maximum number of wells to process.
        rows_per_well (int): Maximum sampled rows per well.
        split (str): Dataset split label.

    Returns:
        pd.DataFrame: Computed DataFrame.
    """
    frames = []
    for path in files[:max_wells]:
        well = well_name_from_horizontal_path(path)
        df = pd.read_csv(path)
        if len(df) > rows_per_well:
            df = df.sample(rows_per_well, random_state=42).sort_index()
        df = df.copy()
        df["well"] = well
        df["split"] = split
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


train_sample = load_horizontal_sample(train_horizontal_files)
print("train sample shape:", train_sample.shape)
display(train_sample.head())

numeric_cols = [
    c
    for c in [
        "MD",
        "X",
        "Y",
        "Z",
        "GR",
        "TVT",
        "TVT_input",
        "ANCC",
        "ASTNU",
        "ASTNL",
        "EGFDU",
        "EGFDL",
        "BUDA",
    ]
    if c in train_sample.columns
]
if numeric_cols:
    numeric_train = train_sample[numeric_cols].apply(
        pd.to_numeric, errors="coerce"
    )
    display(numeric_train.describe().T)

    pearson_corr = numeric_train.corr(method="pearson")
    spearman_corr = numeric_train.corr(method="spearman")

    if "TVT" in numeric_train.columns:
        target_corr = pd.DataFrame(
            {
                "pearson_to_tvt": pearson_corr["TVT"],
                "spearman_to_tvt": spearman_corr["TVT"],
            }
        ).drop(index="TVT", errors="ignore")
        target_corr["abs_pearson"] = target_corr["pearson_to_tvt"].abs()
        target_corr = target_corr.sort_values(
            "abs_pearson", ascending=False
        ).drop(columns="abs_pearson")
        display(target_corr)

    fig, axes = plt.subplots(1, 2, figsize=(22, 8))
    sns.heatmap(
        pearson_corr,
        cmap="viridis",
        annot=True,
        fmt=".2f",
        square=True,
        ax=axes[0],
    )
    axes[0].set_title("Pearson correlation")
    sns.heatmap(
        spearman_corr,
        cmap="viridis",
        annot=True,
        fmt=".2f",
        square=True,
        ax=axes[1],
    )
    axes[1].set_title("Spearman correlation")
    plt.tight_layout()
    plt.show()


In [ ]:
if not train_sample.empty and "TVT" in train_sample.columns:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    sns.histplot(
        pd.to_numeric(train_sample["TVT"], errors="coerce"),
        bins=60,
        ax=axes[0],
        color=VIRIDIS_COLORS[4],
    )
    axes[0].set_title("Train TVT distribution")
    if "GR" in train_sample.columns:
        sns.scatterplot(
            data=train_sample.sample(
                min(len(train_sample), 20000), random_state=42
            ),
            x="GR",
            y="TVT",
            hue="well",
            legend=False,
            s=8,
            alpha=0.35,
            ax=axes[1],
            palette="viridis",
        )
        axes[1].invert_yaxis()
        axes[1].set_title("GR vs TVT sample")
    if "MD" in train_sample.columns:
        sns.scatterplot(
            data=train_sample.sample(
                min(len(train_sample), 20000), random_state=42
            ),
            x="MD",
            y="TVT",
            hue="well",
            legend=False,
            s=8,
            alpha=0.35,
            ax=axes[2],
            palette="viridis",
        )
        axes[2].invert_yaxis()
        axes[2].set_title("MD vs TVT sample")
    plt.tight_layout()
    plt.show()


In [ ]:
if (
    "train_sample" in globals()
    and not train_sample.empty
    and "TVT" in train_sample.columns
):
    numeric_train = train_sample.apply(pd.to_numeric, errors="coerce")
    gr_missing = (
        numeric_train["GR"].isna().mean() if "GR" in numeric_train else np.nan
    )
    tvt_input_missing = (
        numeric_train["TVT_input"].isna().mean()
        if "TVT_input" in numeric_train
        else np.nan
    )
    corr_md_tvt = (
        numeric_train[["MD", "TVT"]].corr().iloc[0, 1]
        if {"MD", "TVT"}.issubset(numeric_train.columns)
        else np.nan
    )
    corr_gr_tvt = (
        numeric_train[["GR", "TVT"]].corr().iloc[0, 1]
        if {"GR", "TVT"}.issubset(numeric_train.columns)
        else np.nan
    )
    show_insights(
        "Target Relationship Insights",
        [
            (
                f"The sampled train table has {fmt_int(len(train_sample))} "
                "rows across "
                f"{fmt_int(train_sample['well'].nunique())} wells."
            ),
            (
                f"`GR` missingness in this sample is {fmt_pct(gr_missing)}, "
                "so missingness flags and interpolation choices may matter."
            ),
            (
                "`TVT_input` missingness in this sample is "
                f"{fmt_pct(tvt_input_missing)}, consistent with a "
                "known-prefix / hidden-suffix task."
            ),
            (
                "Correlation in the sample: `MD` vs `TVT` = "
                f"{fmt_float(corr_md_tvt, 3)}, `GR` vs `TVT` = "
                f"{fmt_float(corr_gr_tvt, 3)}. Correlation alone is "
                "not enough; the useful signal is likely local "
                "log-shape alignment."
            ),
        ],
    )


## 8. EDA Summary

Key findings from the saved EDA run:

- The public sample contains `773` training wells and only `3` public test wells, so public-test behavior should be treated as a smoke test.
- `sample_submission.csv` requests `14,151` predictions across the three public test wells.
- The hidden `TVT_input` suffix is long: median hidden fraction is about `72.7%` in public test and `74.0%` in train.
- Train horizontal files include `TVT` and train-only geology-top columns, while test files expose only inference-time columns (`MD`, `X`, `Y`, `Z`, `GR`, `TVT_input`).
- Train `TVT` range varies substantially by well: median range is about `758 ft`, with a max above `1,250 ft`.
- In the sampled train rows, `GR` missingness is about `32%`, and global `GR`/`TVT` correlation is weak (`-0.052`). The useful signal is likely local log-shape alignment rather than global row-wise correlation.
- A representative PNG image is useful for qualitative QA alongside extracted well metadata, but the current models intentionally remain CSV-feature based.

Modeling handoff:

- Keep train-only geology-top columns out of direct model features.
- Use carry-forward as the baseline to beat.
- Move modeling and submission generation to `notebooks/2_rogii_baseline.ipynb`, where validation is stricter and feature columns are aligned for test-time inference.
- The next high-value feature family is typewell alignment: compare horizontal `GR` windows against typewell `GR` windows indexed by candidate `TVT`.
